# Embed a MicroHH plume into an AVIRIS-4 flight line with SPECTRA

This notebook is used for plume column export, placement, conservative regridding, Lambert-Beer
embedding, matched-filter retrieval and export to the prism contract. The radiance
method runs as three steps (place/regrid, embed, retrieve), so the embedded radiance
can be kept on its own, without the matched filter.

In [ ]:
from pathlib import Path

from spectra.contracts import validate_plume_column, validate_sensor_scene
from spectra.export.prism import to_prism_product, write_product
from spectra.pipeline import (
    EmbeddingSpec,
    embed_radiance_sample,
    embed_sample,
    log_nodes,
    place_and_regrid,
    retrieve_embedded,
    scene_window,
    snapshot_column,
)
from spectra.plume.io import open_plume_column, write_plume_column
from spectra.registry import (
    get_regridder,
    get_retriever,
    get_sensor,
    get_unit_absorption_provider,
)
from spectra.types import Placement, RetrievalSettings

STEM = "M014_240916_FRA_Pau_Methane_3000ft_Line_0001_123559_000_rdn"
RADIANCE = (
    Path("/capstor/store/cscs/c2sm/amrs/ARES/2024-campaign-fr/data/L1_geo/nc/")
    / f"{STEM}.nc"
)  # L1 radiance cube
METADATA = (
    Path("/capstor/store/cscs/c2sm/amrs/ARES/2024-campaign-fr/data/L1_geo/nc/")
    / f"{STEM}_metadata.nc"
)  # PARGE metadata (geolocation, geometry)
SHADOW = (
    Path("/capstor/store/cscs/c2sm/amrs/ARES/2024-campaign-fr/data/L2/shadow_mask")
    / f"{STEM}_shadow.nc"
)  # optional shadow index
PRODUCT = (
    Path("/capstor/store/cscs/c2sm/amrs/ARES/2024-campaign-fr/data/L2/ch4_mf")
    / f"{STEM}_CH4.nc"
)  # optional matched-filter product
RESOURCES = Path(
    "/capstor/store/cscs/c2sm/amrs/ARES/data-unit-abs-spec"
)  # libRadtran/ARTS/solar resources
NOISE_MODEL = Path(
    "/capstor/store/cscs/c2sm/amrs/ARES/emit_noise.txt"
)  # matched-filter noise model
MICROHH_CASE = Path(
    "/capstor/store/cscs/userlab/lp171/smeier/microhh_output/pau_controlled_release.20240918_133000_290kgh_15min_v2_release"
)  # MicroHH case directory
WORK = Path("/ritom/scratch/cscs/smeier/embedding")  # outputs of this notebook
WORK.mkdir(parents=True, exist_ok=True)

## 1. Plume column (Contract A)

The MicroHH processing chain exports a self-describing plume column. The Pau
case has no `datetime_utc` in its `.ini`, so the LES start time is given
explicitly (release start minus 1 h 15 min).

In [ ]:
import microhh_processing_chain.initialisation as init
from microhh_processing_chain.export import export_plume_column

case = init.read_case(case_directory=MICROHH_CASE, scratch_path=MICROHH_CASE.parent)
column_path = WORK / "pau_290kgh.plumecolumn.nc"
if not column_path.exists():
    export_plume_column(
        case, "ch4", start_time="2024-09-18T12:15:00", output_path=column_path
    )
plume = open_plume_column(column_path)
validate_plume_column(plume)

In [ ]:
# Crop to the plume core and re-zero the grid.
crop = plume.sel(x=slice(190, 550), y=slice(340, 580))
x0, y0 = float(crop.x[0]), float(crop.y[0])
crop = crop.assign_coords(x=crop.x - x0, y=crop.y - y0)
crop.attrs.update(
    source_x=plume.attrs["source_x"] - x0,
    source_y=plume.attrs["source_y"] - y0,
    case_id=plume.attrs["case_id"] + "__crop",
    source_id=plume.attrs["case_id"] + "__crop:ch4",
)
crop_path = write_plume_column(
    crop.load(), WORK / "pau_290kgh_crop.plumecolumn.nc", overwrite=True
)
crop["column"].isel(time=120).plot(x="x", y="y", robust=True)

## 2. Scene (Contract B) through the sensor adapter

The adapter opens exactly the files it is given. `record_from_files` builds the
`SceneRecord` (scene id = radiance stem, leakage group `aviris4:<scene_id>`
unless given); many scenes go into a scene table instead
(`adapter.list_scenes("scenes.csv")`, see `docs/recipes.md`).

In [ ]:
adapter = get_sensor("aviris4")
record = adapter.record_from_files(RADIANCE, METADATA, shadow=SHADOW, product=PRODUCT)
scene = adapter.open_scene(record)
validate_sensor_scene(scene, require_lazy=True)
scene

## 3. Placement and plugins

Explicit anchor pixel (along 7000, across 50), upper-left MicroHH corner,
20 degrees clockwise. Omit the indices in the CLI (`spectra embed ... --seed`) to
draw a random anchor whose plume mass stays inside the swath.

In [ ]:
placement = Placement(
    anchor_along=7000,
    anchor_across=50,
    anchor_corner="upper left",
    rotation_deg=-20,
    time_index=120,
)
regridder = get_regridder("polygon")
provider = get_unit_absorption_provider(
    "libradtran_arts",
    uvspec_path=RESOURCES / "spectrum",
    sol_reference_path=RESOURCES / "spectrum",
    arts_path=RESOURCES / "arts",
    xch4_bg=1.93191,
)
retriever = get_retriever("aviris4_mf", noise_model_file=str(NOISE_MODEL))
settings = RetrievalSettings(
    statistics_scope="window",
    band_windows_nm=((1480.0, 1800.0), (2080.0, 2500.0)),
    target_enhancement_ppm=10.0,
    min_n_samples=5000,
    # Idealised background: pixels whose injected column exceeds the threshold
    # are left out of the matched-filter background statistics.
    exclude_injected_from_background=True,
    plume_mask_threshold_kg_m2=0.0,
)
spec = EmbeddingSpec(enhancements_ppm=log_nodes(0.01, 200.0, 12))

## 4. Radiance method, step by step

### 4a. Place and regrid the plume snapshot onto the sensor grid

The result is the injected column on the embedding window, in `kg m-2`, with
the window, placement and regridding provenance in its attributes.

In [ ]:
snapshot = snapshot_column(open_plume_column(crop_path), time_index=120)
injected = place_and_regrid(scene, snapshot, placement, regridder=regridder)
injected.plot(x="across", y="along", robust=True)

### 4b. Lambert-Beer embedding into the radiance (no retrieval)

`embed_radiance_sample` crops the scene to the embedding window, keeps the
bands of `band_windows_nm`, builds the absorption table with the provider and
attenuates the radiance. The result is a `SensorScene` on `(across, along, band)`
carrying `injected_ch4_mass`; it can be written as is (CLI: `--no-retrieval`).

In [ ]:
embedded = embed_radiance_sample(
    scene, injected, provider, spec=spec, band_windows_nm=settings.band_windows_nm
)
validate_sensor_scene(embedded)
write_product(embedded, WORK / "embedded_radiance.nc", overwrite=True)
embedded

In [ ]:
import matplotlib.pyplot as plt

# Band with the strongest plume absorption: lowest mean embedded/original ratio
# over the plume pixels.
window, _ = scene_window(scene, injected)
ratio = (embedded["radiance"] / window["radiance"].sel(band=embedded["band"])).load()
in_plume = embedded["injected_ch4_mass"] > 0
strongest = int(ratio.where(in_plume).mean(("across", "along")).argmin("band"))
wavelength = float(embedded["wavelength"].isel(band=strongest))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
embedded["radiance"].isel(band=strongest).plot(
    ax=axes[0], x="across", y="along", robust=True
)
axes[0].set_title(f"embedded radiance at {wavelength:.0f} nm")
ratio.isel(band=strongest).plot(ax=axes[1], x="across", y="along")
axes[1].set_title("embedded / original radiance")

### 4c. Matched-filter retrieval with the injected plume masked

With `exclude_injected_from_background=True` the retriever receives a
`plume_mask` (injected column above `plume_mask_threshold_kg_m2`) and leaves those
pixels out of the background mean and covariance. `embed_sample(...,
method="radiance")` runs steps 4a-4c in one call.

In [ ]:
product = retrieve_embedded(
    embedded, retriever=retriever, settings=settings, provider=provider, spec=spec
)
prism_product = to_prism_product(product)
write_product(prism_product, WORK / "embedded_radiance_mf.nc", overwrite=True)
prism_product

In [ ]:
plume_mask = prism_product["injected_ch4_mass"] > settings.plume_mask_threshold_kg_m2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
prism_product["injected_ch4_mass"].plot(ax=axes[0], robust=True)
axes[0].set_title("injected column (truth)")
prism_product["ch4_mass"].plot(ax=axes[1], robust=True)
plume_mask.astype(float).plot.contour(
    ax=axes[1], levels=[0.5], colors="k", linewidths=0.8
)
axes[1].set_title("retrieved column, masked plume outlined")

## 5. Product method: add the column onto the existing matched-filter product

In [ ]:
mf_product = adapter.open_product(record)
product_embedded = embed_sample(
    scene,
    snapshot,
    placement,
    method="product",
    regridder=regridder,
    product=mf_product,
)
write_product(
    to_prism_product(product_embedded), WORK / "embedded_product.nc", overwrite=True
)
product_embedded.attrs["regrid_max_relative_mass_error"]

## 6. Equivalent command lines

Every scene file is a flag (`--radiance`, `--metadata`, `--shadow`, `--cloud`,
`--product`, or `--file ROLE=PATH`); `--scene-id` and `--background-group`
override the defaults.

```bash
spectra embed radiance --sensor aviris4 \
  --radiance /path/to/<stem>.nc --metadata /path/to/<stem>_metadata.nc \
  --shadow /path/to/<stem>_shadow.nc \
  --plume pau_290kgh_crop.plumecolumn.nc --time-index 120 \
  --anchor-along 7000 --anchor-across 50 --anchor-corner "upper left" --rotation -20 \
  --uvspec-path /path/to/data-unit-abs-spec/spectrum \
  --sol-reference-path /path/to/data-unit-abs-spec/spectrum \
  --arts-path /path/to/data-unit-abs-spec/arts --xch4-bg 1.93191 \
  --noise-model-file /path/to/emit_noise.txt \
  --band-window 1480 1800 --band-window 2080 2500 --min-n-samples 7000 \
  --exclude-injected-from-background --plume-mask-threshold 0 \
  --save-path embedded_radiance_mf.nc --prism
# embedded radiance only (steps 4a-4b, no matched filter, no --prism)
spectra embed radiance <same scene, plume, placement and provider flags> \
  --band-window 1480 1800 --band-window 2080 2500 \
  --no-retrieval --save-path embedded_radiance.nc
spectra embed product --sensor aviris4 \
  --radiance /path/to/<stem>.nc --metadata /path/to/<stem>_metadata.nc \
  --product /path/to/<stem>_CH4.nc \
  --plume pau_290kgh_crop.plumecolumn.nc --time-index 120 \
  --anchor-along 7000 --anchor-across 50 --anchor-corner "upper left" --rotation -20 \
  --save-path embedded_product.nc --prism
```

Datasets with many samples are generated from a recipe and a scene table
(`spectra catalog build --sensor aviris4 --scenes scenes.csv --out catalog/`,
then `spectra generate recipe.yaml --out <dir> --dry-run`, `spectra generate`,
`spectra finalize` and `prism-compose-dataset <dir>/pairings.csv ...`).